# 09 — Claude Haiku Grading Evaluator

Exact Python port of the Chrome extension grading pipeline for offline evaluation.

Replicates **every step** of `claudeVision.ts` + `cvDetectors.ts`:
1. CV detectors — Laplacian blur, glare fraction, brightness (same constants as production)
2. Image resize — longest edge ≤ 1024 px, JPEG q85
3. Prompt injection — CV measurements prepended verbatim
4. Claude Haiku call — same model, max_tokens, system/user prompts
5. Distribution normalisation

Use this to:
- Test new prompts before deploying
- Evaluate consistency across multiple images of the same card
- Build a labelled dataset for accuracy benchmarking
- Debug why a particular card got an unexpected grade

In [ ]:
# Run once, then comment out
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'anthropic', 'Pillow', 'numpy', 'opencv-python', 'requests',
    'matplotlib', 'pandas', 'tqdm'])
print('✅ Dependencies ready')

In [ ]:
import os, io, re, json, base64, textwrap, warnings
from pathlib import Path

import numpy as np
import cv2
import requests
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import anthropic
from tqdm.auto import tqdm
from IPython.display import display as ipy_display, HTML

warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi':       120,
    'axes.facecolor':   '#161b22',
    'figure.facecolor': '#0d1117',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('✅ Imports OK')

## Configuration

Set your Anthropic API key. The notebook reads `ANTHROPIC_API_KEY` from the environment, or you can paste it directly.

In [ ]:
# ── API Key ───────────────────────────────────────────────────────────────────
API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not API_KEY:
    API_KEY = input('Paste your Anthropic API key: ').strip()
assert API_KEY, 'API key required'
print(f'✅ API key loaded ({API_KEY[:8]}...)')

## CV Detectors + Side Classifier

Exact Python port of `cvDetectors.ts`. Same constants, same algorithm.

| Constant | Value | Meaning |
|---|---|---|
| `CANONICAL_W` | 384 | Resize width for CV analysis |
| `CANONICAL_H` | 544 | Resize height for CV analysis |
| `BLUR_THRESHOLD` | 40 | Laplacian variance below = blurry |
| `GLARE_THRESHOLD` | 0.05 | Fraction of pixels ≥ 245 above = significant glare |
| `BACK_HUE_MIN/MAX` | 195–220° | HSV hue band for Pokémon card back blue |
| `BACK_SAT_MIN` | 0.50 | Min saturation for back-blue pixels |
| `BACK_PIXEL_THRESHOLD` | 0.25 | >25% matching pixels → classify as back |

In [ ]:
# ── Constants — must match cvDetectors.ts ─────────────────────────────────────
CANONICAL_W     = 384
CANONICAL_H     = 544
BLUR_THRESHOLD  = 40
GLARE_THRESHOLD = 0.05

# Corner analysis
CORNER_W_PX          = 46
CORNER_H_PX          = 55
WHITE_THRESH          = 215
WHITENING_THRESHOLD   = 0.07

# Surface roughness (border-band Sobel)
BORDER_BAND_W              = 55
BORDER_BAND_H              = 70
SCRATCH_GRADIENT_THRESHOLD = 35
SCRATCH_SEVERITY = {'none': 0, 'light': 200, 'moderate': 600, 'heavy': 1400}

# Side classifier HSV thresholds
BACK_HUE_MIN         = 195
BACK_HUE_MAX         = 220
BACK_SAT_MIN         = 0.50
BACK_VAL_MIN         = 0.25
BACK_PIXEL_THRESHOLD = 0.25

# Centering (perspective warp pipeline)
CARD_ASPECT_W  = 500   # canonical warped card width  (px)
CARD_ASPECT_H  = 700   # canonical warped card height (px)
BORDER_SCAN_W  = 50    # scan width for border measurement (px from each edge)
BORDER_SCAN_H  = 60    # scan height for border measurement


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — image quality
# ══════════════════════════════════════════════════════════════════════════════

def _laplacian_variance(gray: np.ndarray) -> float:
    lap = cv2.Laplacian(gray.astype(np.float64), cv2.CV_64F)
    return float(lap.var())


def _analyse_image_array(img_rgb: np.ndarray) -> dict:
    """Blur / glare / brightness on the canonical resized image."""
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H),
                           interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    total            = gray.size
    brightness_mean  = float(gray.mean())
    brightness_std   = float(gray.std())
    glare_count      = int(np.sum(gray >= 245))
    glare_fraction   = glare_count / total
    blur_score       = _laplacian_variance(gray)
    is_blurry        = blur_score < BLUR_THRESHOLD
    has_glare        = glare_fraction > GLARE_THRESHOLD

    cv_issues = []
    if is_blurry:
        cv_issues.append(f'blurry image (score {blur_score:.1f}, need ≥{BLUR_THRESHOLD}) — reduce grade confidence')
    if has_glare:
        cv_issues.append(f'surface glare on {glare_fraction*100:.1f}% of card area — may hide scratches or haze')
    if brightness_mean < 50:
        cv_issues.append('very dark image — poor lighting may obscure surface defects')
    elif brightness_mean > 220 and not has_glare:
        cv_issues.append('overexposed image — highlight detail may be lost')

    return dict(blur_score=round(blur_score,1), glare_fraction=round(glare_fraction,4),
                brightness_mean=round(brightness_mean,1), brightness_std=round(brightness_std,1),
                is_blurry=is_blurry, has_glare=has_glare, cv_issues=cv_issues)


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2a — corner whitening
# ══════════════════════════════════════════════════════════════════════════════

def _corner_patch(gray_canonical: np.ndarray, r0: int, c0: int, ph: int, pw: int) -> dict:
    patch          = gray_canonical[r0:r0+ph, c0:c0+pw]
    brightness     = float(patch.mean())
    white_fraction = float((patch > WHITE_THRESH).mean())
    return dict(brightness=round(brightness,1),
                white_fraction=round(white_fraction,4),
                whitening=white_fraction > WHITENING_THRESHOLD)


def analyse_corners(img_rgb: np.ndarray) -> dict:
    """
    Per-corner whitening analysis on the canonical (384×544) grayscale image.

    Matches cvDetectors.ts computeCornerAnalysis().
    Each corner is a CORNER_W_PX × CORNER_H_PX patch.
    Returns TL/TR/BL/BR CornerPatch dicts + any_whitening + whitening_corners.
    """
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray      = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY)
    H, W      = gray.shape
    pw, ph    = CORNER_W_PX, CORNER_H_PX

    patches = {
        'TL': _corner_patch(gray, 0,      0,      ph, pw),
        'TR': _corner_patch(gray, 0,      W - pw, ph, pw),
        'BL': _corner_patch(gray, H - ph, 0,      ph, pw),
        'BR': _corner_patch(gray, H - ph, W - pw, ph, pw),
    }
    whitening_corners = [k for k, v in patches.items() if v['whitening']]
    return dict(**patches,
                any_whitening=len(whitening_corners) > 0,
                whitening_corners=whitening_corners)


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2b — surface roughness / scratch indicator (border-band Sobel)
# ══════════════════════════════════════════════════════════════════════════════

def analyse_scratch_indicator(img_rgb: np.ndarray) -> dict:
    """
    Sobel gradient magnitude in the card-border band.

    The card's printed border should be smooth; physical scratches and print
    lines show up as unexpected high-gradient pixels. Artwork is excluded to
    avoid false positives from the card illustration.

    Matches cvDetectors.ts computeScratchIndicator().
    Severity thresholds: none <200, light <600, moderate <1400, heavy ≥1400.
    """
    canonical = cv2.resize(img_rgb, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray      = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    H, W      = gray.shape

    # Build border-band mask (perimeter band, excluding inner artwork)
    mask      = np.zeros((H, W), dtype=bool)
    mask[:BORDER_BAND_H, :]  = True   # top
    mask[-BORDER_BAND_H:, :] = True   # bottom
    mask[:, :BORDER_BAND_W]  = True   # left
    mask[:, -BORDER_BAND_W:] = True   # right

    sobel_x   = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y   = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

    edge_count = int((magnitude[mask] > SCRATCH_GRADIENT_THRESHOLD).sum())

    t = SCRATCH_SEVERITY
    severity = ('heavy'    if edge_count >= t['heavy']    else
                'moderate' if edge_count >= t['moderate'] else
                'light'    if edge_count >= t['light']    else 'none')

    return dict(edge_pixel_count=edge_count, severity=severity)


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2c — centering (perspective warp + border measurement)
# ══════════════════════════════════════════════════════════════════════════════

def _find_card_quad(img_rgb: np.ndarray) -> np.ndarray | None:
    """
    Locate the card quadrilateral using Canny + contour approximation.
    Returns a (4, 1, 2) int32 array (OpenCV contour format) or None.
    """
    gray    = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Adaptive Canny: lower threshold at 40% of Otsu's value
    otsu, _  = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    edges    = cv2.Canny(blurred, otsu * 0.4, otsu)
    # Dilate to close small gaps in the card border
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges    = cv2.dilate(edges, kernel, iterations=1)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    img_area    = img_rgb.shape[0] * img_rgb.shape[1]
    best        = None
    best_area   = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < img_area * 0.15:   # ignore tiny contours
            continue
        peri   = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4 and area > best_area:
            best      = approx
            best_area = area

    return best


def _order_quad(pts: np.ndarray) -> np.ndarray:
    """Order 4 points: TL, TR, BR, BL."""
    pts   = pts.reshape(4, 2).astype(np.float32)
    s     = pts.sum(axis=1)
    diff  = np.diff(pts, axis=1).ravel()
    tl    = pts[np.argmin(s)]
    br    = pts[np.argmax(s)]
    tr    = pts[np.argmin(diff)]
    bl    = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def perspective_warp(img_rgb: np.ndarray) -> tuple[np.ndarray | None, bool]:
    """
    Detect card quad and warp to a canonical CARD_ASPECT_W × CARD_ASPECT_H view.

    Returns (warped_img, warp_applied).
    If quad detection fails, returns (None, False) — caller falls back to
    uncorrected measurements.
    """
    quad = _find_card_quad(img_rgb)
    if quad is None:
        return None, False

    src = _order_quad(quad)
    dst = np.array([[0, 0],
                    [CARD_ASPECT_W - 1, 0],
                    [CARD_ASPECT_W - 1, CARD_ASPECT_H - 1],
                    [0, CARD_ASPECT_H - 1]], dtype=np.float32)

    M       = cv2.getPerspectiveTransform(src, dst)
    warped  = cv2.warpPerspective(img_rgb, M, (CARD_ASPECT_W, CARD_ASPECT_H))
    return warped, True


def _measure_border(gray: np.ndarray, axis: int, from_end: bool,
                    band: int, threshold: float) -> int:
    """
    Scan inward along `axis` to find the first row/col where variance exceeds
    `threshold` — proxy for where the card border transitions to artwork.

    axis=0 scans rows (top/bottom), axis=1 scans cols (left/right).
    from_end=True scans from the far edge inward.
    Returns the detected border width in pixels.
    """
    n = gray.shape[axis]
    indices = range(n - 1, -1, -1) if from_end else range(n)
    for i, idx in enumerate(indices):
        row_or_col = gray[idx, :] if axis == 0 else gray[:, idx]
        if float(row_or_col.std()) > threshold:
            return i
    return band  # fallback: full band


def measure_centering(img_rgb: np.ndarray) -> dict:
    """
    Measure card centering via perspective warp + border scan.

    Pipeline:
      1. Detect card quadrilateral (Canny + contour)
      2. Warp to flat canonical view (500×700)
      3. Scan each side inward until content variance rises
      4. Compute L/R and T/B border ratios

    Returns dict with lr_ratio, tb_ratio, border_px, warp_applied, reliable.
    reliable=False when warp detection failed (measurements from raw image).
    """
    warped, warp_ok = perspective_warp(img_rgb)
    work_img = warped if warp_ok else cv2.resize(
        img_rgb, (CARD_ASPECT_W, CARD_ASPECT_H), interpolation=cv2.INTER_LANCZOS4)

    gray      = cv2.cvtColor(work_img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    H, W      = gray.shape
    VAR_THRESH = 8.0   # std-dev threshold: above = artwork content

    top    = _measure_border(gray, 0, False, BORDER_SCAN_H, VAR_THRESH)
    bottom = _measure_border(gray, 0, True,  BORDER_SCAN_H, VAR_THRESH)
    left   = _measure_border(gray, 1, False, BORDER_SCAN_W, VAR_THRESH)
    right  = _measure_border(gray, 1, True,  BORDER_SCAN_W, VAR_THRESH)

    tb_sum = max(top + bottom, 1)
    lr_sum = max(left + right, 1)

    lr_ratio = f"{round(left/lr_sum*100)}/{round(right/lr_sum*100)}"
    tb_ratio = f"{round(top/tb_sum*100)}/{round(bottom/tb_sum*100)}"

    return dict(
        lr_ratio   = lr_ratio,
        tb_ratio   = tb_ratio,
        border_px  = dict(top=top, bottom=bottom, left=left, right=right),
        warp_applied = warp_ok,
        reliable   = warp_ok,   # measurements from warped image are much more reliable
    )


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3 — side classifier
# ══════════════════════════════════════════════════════════════════════════════

def classify_card_side(img_rgb: np.ndarray) -> str:
    """HSV blue-band classifier for Pokémon card back. Port of classifyCardSide()."""
    try:
        small = cv2.resize(img_rgb, (128, 128), interpolation=cv2.INTER_AREA)
        hsv   = cv2.cvtColor(small, cv2.COLOR_RGB2HSV).astype(np.float32)
        h     = hsv[:, :, 0] * 2.0
        s     = hsv[:, :, 1] / 255.0
        v     = hsv[:, :, 2] / 255.0
        mask  = (h >= BACK_HUE_MIN) & (h <= BACK_HUE_MAX) & (s >= BACK_SAT_MIN) & (v >= BACK_VAL_MIN)
        return 'back' if mask.sum() / (128 * 128) >= BACK_PIXEL_THRESHOLD else 'front'
    except Exception as exc:
        print(f'  classify_card_side failed: {exc}')
        return 'unknown'


# ══════════════════════════════════════════════════════════════════════════════
#  Prompt formatters
# ══════════════════════════════════════════════════════════════════════════════

def format_cv_section(cv: dict | None) -> str:
    if cv is None:
        return ''
    issue_str  = '\n'.join(f'  • {s}' for s in cv['cv_issues']) if cv['cv_issues'] else '  • none detected'
    blur_note  = '\nNOTE: Image is blurry — lower confidence, flag in issues.'  if cv['is_blurry'] else ''
    glare_note = '\nNOTE: Significant glare — surface haze or scratches may be hidden.' if cv['has_glare'] else ''
    return (f'─── CV MEASUREMENTS (pixel analysis, run before this prompt) ───\n'
            f'Blur score   : {cv["blur_score"]}  (threshold ≥ {BLUR_THRESHOLD} = acceptably sharp)\n'
            f'Glare        : {cv["glare_fraction"]*100:.1f}% pixels overexposed  (≤ 5% acceptable)\n'
            f'Brightness   : mean {cv["brightness_mean"]} / std {cv["brightness_std"]}  (0–255 scale)\n'
            f'CV issues    :\n{issue_str}{blur_note}{glare_note}\n\n')


def format_corner_section(corners: dict | None) -> str:
    if corners is None:
        return ''
    def fmt(p):
        w = f"  ⚠ WHITENING" if p['whitening'] else ''
        return f"brightness {p['brightness']}, white {p['white_fraction']*100:.1f}%{w}"
    note = (f"NOTE: Whitening detected at {', '.join(corners['whitening_corners'])} — "
            f"these corners cannot grade PSA 10; likely caps at PSA 8 or below."
            if corners['any_whitening']
            else "NOTE: No corner whitening detected in pixel analysis.")
    return (f'─── CV CORNER ANALYSIS (pixel measurement) ───\n'
            f'  TL: {fmt(corners["TL"])}\n'
            f'  TR: {fmt(corners["TR"])}\n'
            f'  BL: {fmt(corners["BL"])}\n'
            f'  BR: {fmt(corners["BR"])}\n'
            f'{note}\n\n')


def format_scratch_section(scratch: dict | None) -> str:
    if scratch is None:
        return ''
    note = ('NOTE: Elevated gradient features — possible scratches or print lines. Inspect surface carefully.'
            if scratch['severity'] != 'none'
            else 'NOTE: Border band appears smooth — no strong gradient spikes detected.')
    return (f'─── CV SURFACE ROUGHNESS (border-band gradient, approximate) ───\n'
            f'  Edge-pixel count: {scratch["edge_pixel_count"]}  '
            f'Severity: {scratch["severity"].upper()}\n'
            f'{note}\n\n')


def format_centering_section(centering: dict | None) -> str:
    if centering is None:
        return ''
    method = 'perspective-warped image' if centering['warp_applied'] else 'uncorrected image (warp failed)'
    px     = centering['border_px']
    reliable_note = '' if centering['reliable'] else '\nNOTE: Warp detection failed — centering is approximate.'
    return (f'─── CV CENTERING MEASUREMENT ({method}) ───\n'
            f'  L/R: {centering["lr_ratio"]}  T/B: {centering["tb_ratio"]}\n'
            f'  Border px: top={px["top"]} bottom={px["bottom"]} '
            f'left={px["left"]} right={px["right"]}\n'
            f'NOTE: Use this as ground truth for centering. '
            f'Deviations from 50/50 directly affect PSA grade ceiling.{reliable_note}\n\n')


def format_side_labels(sides: list[str]) -> str:
    if all(s == 'unknown' for s in sides):
        return ''
    lines = '\n'.join(f'  Image {i+1}: {s.upper()}' for i, s in enumerate(sides))
    return (f'─── IMAGE SIDE CLASSIFICATION (pre-classified by pixel analysis) ───\n'
            f'{lines}\n'
            f'NOTE: Trust these labels — do NOT reclassify images yourself.\n\n')


# ══════════════════════════════════════════════════════════════════════════════
#  Helper loaders (used by grade_with_claude and visualisers)
# ══════════════════════════════════════════════════════════════════════════════

def _fetch_buffer(url: str) -> np.ndarray | None:
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        return np.array(Image.open(io.BytesIO(resp.content)).convert('RGB'))
    except Exception as exc:
        print(f'  fetch failed: {url[:60]}... ({exc})')
        return None


def _load_buffer(path: str) -> np.ndarray | None:
    try:
        return np.array(Image.open(path).convert('RGB'))
    except Exception as exc:
        print(f'  load failed: {path} ({exc})')
        return None


def run_cv_detectors(image_urls: list[str]) -> dict | None:
    """Phase 1 only (quality). Tries first 2 URLs."""
    for url in image_urls[:2]:
        try:
            img = _fetch_buffer(url)
            if img is not None:
                result = _analyse_image_array(img)
                result['_source_url'] = url
                return result
        except Exception as exc:
            print(f'  CV: skipping {url[:70]}... ({exc})')
    return None


def run_cv_detectors_local(image_path: str) -> dict | None:
    try:
        img = _load_buffer(image_path)
        if img is None:
            return None
        result = _analyse_image_array(img)
        result['_source_url'] = image_path
        return result
    except Exception as exc:
        print(f'  CV local: failed ({exc})')
        return None


print('✅ CV detectors + structural analysis + side classifier ready')

### Side Classifier Test

Visualise the HSV blue-band mask on your local images to verify the classifier is working correctly before running a full grading call.

In [ ]:
def visualise_side_classifier(image_paths: list[str] | None = None,
                               image_urls:  list[str] | None = None) -> None:
    """
    Show each image alongside its HSV blue-band mask and classification result.
    Useful for tuning thresholds or debugging misclassifications.
    """
    sources = image_paths or image_urls or []
    imgs: list[np.ndarray] = []
    for src in sources:
        arr = _load_buffer(src) if image_paths else _fetch_buffer(src)
        if arr is not None:
            imgs.append(arr)

    if not imgs:
        print('No images loaded')
        return

    n = len(imgs)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 7))
    if n == 1:
        axes = axes.reshape(2, 1)

    for col, img in enumerate(imgs):
        small = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
        hsv   = cv2.cvtColor(small, cv2.COLOR_RGB2HSV).astype(np.float32)
        h = hsv[:, :, 0] * 2.0
        s = hsv[:, :, 1] / 255.0
        v = hsv[:, :, 2] / 255.0

        mask = (
            (h >= BACK_HUE_MIN) & (h <= BACK_HUE_MAX) &
            (s >= BACK_SAT_MIN) &
            (v >= BACK_VAL_MIN)
        )
        blue_frac = mask.sum() / (128 * 128)
        label     = classify_card_side(img)
        colour    = '#10b981' if label == 'back' else '#f97316'

        # Top row: original
        axes[0, col].imshow(img)
        axes[0, col].set_title(
            f'{label.upper()}  ({blue_frac*100:.1f}% blue)',
            color=colour, fontsize=11, fontweight='bold')
        axes[0, col].axis('off')

        # Bottom row: mask overlay
        overlay = small.copy()
        overlay[mask] = [0, 200, 100]
        axes[1, col].imshow(overlay)
        axes[1, col].set_title('Blue-band mask', color='white', fontsize=9)
        axes[1, col].axis('off')

    plt.suptitle(
        f'Threshold: >{BACK_PIXEL_THRESHOLD*100:.0f}% blue pixels → BACK',
        color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


# ── Run the test on your local images ─────────────────────────────────────────
# Change to image_urls=['url1','url2'] to test with eBay URLs instead.
TEST_CLASSIFY_PATHS = ['image0_front.jpeg', 'image0_back.jpeg']

if all(Path(p).exists() for p in TEST_CLASSIFY_PATHS):
    visualise_side_classifier(image_paths=TEST_CLASSIFY_PATHS)
else:
    print('Local test images not found — set TEST_CLASSIFY_PATHS or use image_urls')

### Phase 2 CV Visualiser — Corners, Centering, Scratch

Run this on any local image to see the full structural analysis before sending to Claude.

In [ ]:
def visualise_cv_pipeline(image_path: str | None = None,
                           image_url:  str | None = None) -> None:
    """
    Full Phase 2 CV visualisation for a single card image:
    - Corner patches with whitening overlay
    - Border-band Sobel gradient mask
    - Perspective-warped card with centering border lines
    """
    img = _load_buffer(image_path) if image_path else _fetch_buffer(image_url)
    if img is None:
        print('Failed to load image')
        return

    # Run all Phase 2 detectors
    corners   = analyse_corners(img)
    scratch   = analyse_scratch_indicator(img)
    centering = measure_centering(img)
    side      = classify_card_side(img)

    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#0d1117')

    # ── Row 1: Original | Warped | Sobel border mask ──────────────────────────
    ax1 = fig.add_subplot(2, 3, 1)
    ax1.imshow(img)
    ax1.set_title(f'Original  [{side.upper()}]', color='white')
    ax1.axis('off')

    warped, warp_ok = perspective_warp(img)
    ax2 = fig.add_subplot(2, 3, 2)
    if warp_ok and warped is not None:
        ax2.imshow(warped)
        # Draw centering lines
        c  = centering
        px = c['border_px']
        H, W = warped.shape[:2]
        ax2.axhline(px['top'],        color='#10b981', lw=1.5, alpha=0.8, label=f"top={px['top']}")
        ax2.axhline(H - px['bottom'], color='#10b981', lw=1.5, alpha=0.8, label=f"bot={px['bottom']}")
        ax2.axvline(px['left'],       color='#6366f1', lw=1.5, alpha=0.8, label=f"left={px['left']}")
        ax2.axvline(W - px['right'],  color='#6366f1', lw=1.5, alpha=0.8, label=f"right={px['right']}")
        ax2.set_title(f"Warped  L/R {c['lr_ratio']}  T/B {c['tb_ratio']}", color='#10b981')
    else:
        ax2.text(0.5, 0.5, 'Warp failed\n(no clear card quad found)',
                 ha='center', va='center', color='#f97316', fontsize=11,
                 transform=ax2.transAxes)
        ax2.set_title('Centering (warp failed)', color='#f97316')
    ax2.axis('off')

    # Sobel border-band mask
    canonical = cv2.resize(img, (CANONICAL_W, CANONICAL_H), interpolation=cv2.INTER_LANCZOS4)
    gray_c    = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY).astype(np.float32)
    sx        = cv2.Sobel(gray_c, cv2.CV_32F, 1, 0, ksize=3)
    sy        = cv2.Sobel(gray_c, cv2.CV_32F, 0, 1, ksize=3)
    mag       = np.sqrt(sx**2 + sy**2)
    H_c, W_c  = canonical.shape[:2]
    band_mask = np.zeros((H_c, W_c), dtype=bool)
    band_mask[:BORDER_BAND_H, :]  = True
    band_mask[-BORDER_BAND_H:, :] = True
    band_mask[:, :BORDER_BAND_W]  = True
    band_mask[:, -BORDER_BAND_W:] = True

    viz_sobel = canonical.copy()
    edge_mask = (mag > SCRATCH_GRADIENT_THRESHOLD) & band_mask
    viz_sobel[edge_mask]          = [255, 80, 80]   # red = high gradient in border
    viz_sobel[band_mask & ~edge_mask] = (viz_sobel[band_mask & ~edge_mask] * 0.5 + np.array([0, 80, 150]) * 0.5).astype(np.uint8)

    ax3 = fig.add_subplot(2, 3, 3)
    ax3.imshow(viz_sobel)
    ax3.set_title(f"Scratch indicator  {scratch['severity'].upper()}  ({scratch['edge_pixel_count']} px)",
                  color='#ef4444' if scratch['severity'] != 'none' else '#10b981')
    ax3.axis('off')

    # ── Row 2: Four corner patches ────────────────────────────────────────────
    corner_positions = [('TL', 2, 3, 4), ('TR', 2, 3, 5), ('BL', 2, 3, 6)]  # 3 fit; BR in title
    gray_can_uint8 = cv2.cvtColor(canonical, cv2.COLOR_RGB2GRAY)
    corner_imgs = {
        'TL': canonical[:CORNER_H_PX, :CORNER_W_PX],
        'TR': canonical[:CORNER_H_PX, -CORNER_W_PX:],
        'BL': canonical[-CORNER_H_PX:, :CORNER_W_PX],
        'BR': canonical[-CORNER_H_PX:, -CORNER_W_PX:],
    }
    for col_idx, (name, _, _, subplot_idx) in enumerate(corner_positions):
        ax = fig.add_subplot(2, 3, subplot_idx)
        patch_img = corner_imgs[name].copy()
        p = corners[name]
        # Highlight white pixels
        white_mask = cv2.cvtColor(patch_img, cv2.COLOR_RGB2GRAY) > WHITE_THRESH
        patch_img[white_mask] = [255, 100, 100] if p['whitening'] else [100, 200, 100]
        ax.imshow(patch_img)
        color = '#ef4444' if p['whitening'] else '#10b981'
        ax.set_title(f"{name}  {p['brightness']:.0f}bri  {p['white_fraction']*100:.1f}%white"
                     + (' ⚠' if p['whitening'] else ''), color=color, fontsize=9)
        ax.axis('off')

    # BR in a text box
    ax_br = fig.add_subplot(2, 3, 6) if len(corner_positions) < 4 else None
    if ax_br:
        br = corners['BR']
        patch_img = corner_imgs['BR'].copy()
        white_mask = cv2.cvtColor(patch_img, cv2.COLOR_RGB2GRAY) > WHITE_THRESH
        patch_img[white_mask] = [255, 100, 100] if br['whitening'] else [100, 200, 100]
        ax_br.imshow(patch_img)
        color = '#ef4444' if br['whitening'] else '#10b981'
        ax_br.set_title(f"BR  {br['brightness']:.0f}bri  {br['white_fraction']*100:.1f}%white"
                        + (' ⚠' if br['whitening'] else ''), color=color, fontsize=9)
        ax_br.axis('off')

    summary = (f"Corners: {'⚠ ' + ', '.join(corners['whitening_corners']) if corners['any_whitening'] else '✓ clean'}  |  "
               f"Centering: L/R {centering['lr_ratio']}  T/B {centering['tb_ratio']}  |  "
               f"Surface: {scratch['severity']}")
    fig.suptitle(summary, color='white', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

    # Print injected CV sections
    print('\n' + '─'*65)
    print('PROMPT SECTIONS THAT WOULD BE INJECTED:')
    print('─'*65)
    print(format_corner_section(corners), end='')
    print(format_scratch_section(scratch), end='')
    print(format_centering_section(centering), end='')


# ── Run on your local images ──────────────────────────────────────────────────
VIZ_PATH = 'image0_front.jpeg'   # change to your image

if Path(VIZ_PATH).exists():
    visualise_cv_pipeline(image_path=VIZ_PATH)
else:
    print(f'Set VIZ_PATH to an existing image file (tried: {VIZ_PATH})')

## Image Helpers

Port of `fetchResized()` from `claudeVision.ts` — downloads and resizes the primary image to ≤ 1024 px for fewer Claude input tokens.

In [ ]:
def fetch_resized(url: str, max_size: int = 1024, quality: int = 85) -> dict | None:
    """Port of fetchResized() from claudeVision.ts.

    Downloads url, resizes longest edge to ≤ max_size (no upscaling),
    encodes as JPEG base64.  Returns None on any failure.
    """
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        return {
            'base64':     base64.b64encode(buf.getvalue()).decode(),
            'media_type': 'image/jpeg',
            'size':       img.size,
        }
    except Exception as exc:
        print(f'  resize failed: {exc}')
        return None


def load_local_image(path: str, max_size: int = 1024, quality: int = 85) -> dict | None:
    """Same as fetch_resized but from a local file path."""
    try:
        img = Image.open(path).convert('RGB')
        w, h = img.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        return {
            'base64':     base64.b64encode(buf.getvalue()).decode(),
            'media_type': 'image/jpeg',
            'size':       img.size,
        }
    except Exception as exc:
        print(f'  local load failed: {exc}')
        return None


def show_images(image_urls: list[str] | None = None,
                local_paths: list[str] | None = None,
                max_show: int = 6):
    """Display thumbnail grid of the test images."""
    sources = []
    if image_urls:
        for url in image_urls[:max_show]:
            try:
                resp = requests.get(url.replace('s-l1600', 's-l300'), timeout=6)
                sources.append(Image.open(io.BytesIO(resp.content)).convert('RGB'))
            except Exception:
                pass
    if local_paths:
        for p in local_paths[:max_show]:
            try:
                sources.append(Image.open(p).convert('RGB'))
            except Exception:
                pass

    if not sources:
        print('No images to display')
        return

    n = len(sources)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, img in zip(axes, sources):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'{n} image(s)', color='white', fontsize=11)
    plt.tight_layout()
    plt.show()


print('\u2705 Image helpers ready')

## Prompts

Copied verbatim from `src/lib/grading/claudeVision.ts`. **Edit these here to test prompt changes before deploying.**

In [ ]:
SYSTEM_PROMPT = """You are an expert PSA pre-grading assistant specializing in Pokémon trading cards.

Your task is to estimate the plausible PSA grade range supported by the visible evidence. You are not guaranteeing a final PSA grade.

Core principles:
- Base your assessment only on what is actually visible in the images.
- Never assume unseen areas are clean.
- If glare, blur, angle, cropping, sleeve reflections, compression, or low resolution hide details, reduce confidence and mention the limitation.
- When evidence is incomplete, prefer a wider grade range and lower confidence.
- Distinguish between: (1) visible defects and (2) unknowns caused by image limitations.

Evaluate in this order:
1. IMAGE CLASSIFICATION — label each image as "front", "back", or "other" based on visible content. The front shows the card artwork; the back shows the Pokémon logo / back design.
2. FRONT ANALYSIS — if a front image is present, assess separately:
   - Centering: estimate L/R and T/B border split
   - Corners: TL, TR, BL, BR — whitening, fraying, rounding
   - Edges: top, bottom, left, right — nicks, chips, whitening
   - Surface: scratches, print lines, holo damage, staining, gloss loss
3. BACK ANALYSIS — if a back image is present, assess separately:
   - Centering: estimate L/R and T/B border split
   - Corners: TL, TR, BL, BR
   - Edges: top, bottom, left, right
   - Surface: print quality, scratches, staining
4. COMBINED GRADE — derive the final grade_estimate and combined issues from the worst-case evidence across both sides.
5. CARD IDENTITY — identify name/set/year/number from the front image only.

PSA grade guidance:
- PSA 10 Gem Mint: near-perfect, centering ~55/45 or better on front, four sharp corners, no visible defects, full gloss
- PSA 9  Mint: slight visible wear allowed, minor issues possible, strong overall presentation
- PSA 8  NM-MT: light corner/edge wear, possible very light scratches or print issues
- PSA 7  NM: visible but not severe corner/edge wear, possible light scratches or minor print defects
- PSA 6  EX-MT and below: increasingly obvious wear, scratches, edge damage, staining, creasing, or major defects

Front-only rule (CRITICAL):
- If no back image is identifiable, set analysis_mode to "front_only"
- Set back_analysis.assessable to false and leave all back_analysis issue arrays empty
- Set grade_estimate.confidence to "low" regardless of front image quality
- Set grade_estimate.limiting_factor to "front_only"
- Add exactly this string to grading_decision.caveats: "Back image not provided — rear corner whitening, edge wear, and back centering cannot be assessed. Grade confidence is limited to low."
- Do NOT speculate about the back condition

Other special instructions:
- Do not double-count the same defect across front and back.
- Vintage Pokémon cards (1999–2003) should be judged with awareness of era, but never use era to override visible evidence.
- Holo surface must be treated conservatively when glare or angle prevents reliable inspection.
- Do not overclaim PSA 9 or PSA 10 when image quality is limited.

Respond ONLY with one valid JSON object. No markdown, no prose outside JSON.

Every "issues" object must use exactly these keys: centering, corners, edges, surface, other.
Each maps to an array of strings. Use [] when nothing is visible for that category.

Allowed values:
- "analysis_mode": "front_only" or "front_back"
- "image_quality.status": "good", "partial", or "poor"
- "card_identity.confidence": "high", "medium", or "low"
- "grade_estimate.confidence": "high", "medium", or "low"
- "grade_estimate.limiting_factor": "front_only", "image_quality", "visible_damage", or null
- "grading_decision.gradable_candidate": "yes", "maybe", or "no"
- "front_analysis.assessable" / "back_analysis.assessable": true or false

If a card_identity field is uncertain, use null rather than guessing."""


USER_PROMPT = """Analyze this Pokémon card from the provided image set and return JSON with EXACTLY this structure.

Example when both front and back are present:
{
  "analysis_mode": "front_back",
  "card_identity": {
    "name": "Charizard", "set": "Base Set", "year": "1999", "number": "4", "confidence": "high"
  },
  "image_quality": {
    "status": "good", "warnings": [],
    "front_present": true, "back_present": true,
    "centering_visible": true, "corners_visible": true, "edges_visible": true, "surface_visible": true
  },
  "front_analysis": {
    "assessable": true,
    "centering": "58/42 L/R, 54/46 T/B",
    "issues": {
      "centering": ["Slightly left-heavy, approximately 58/42 L/R"],
      "corners":   ["Light whitening on top-right corner"],
      "edges":     [],
      "surface":   ["Faint holo scratches visible at angle"],
      "other":     []
    }
  },
  "back_analysis": {
    "assessable": true,
    "centering": "50/50 L/R, 51/49 T/B",
    "issues": {
      "centering": [],
      "corners":   [],
      "edges":     ["Minor whitening on bottom edge"],
      "surface":   [],
      "other":     []
    }
  },
  "grade_estimate": {
    "grade_range": "PSA 7-8", "confidence": "high", "limiting_factor": "visible_damage",
    "distribution": {
      "1": 0.00, "2": 0.00, "3": 0.01, "4": 0.02, "5": 0.04,
      "6": 0.08, "7": 0.30, "8": 0.40, "9": 0.12, "10": 0.03
    }
  },
  "issues": {
    "centering": ["Slightly left-heavy front (58/42 L/R)"],
    "corners":   ["Light whitening on top-right corner (front)"],
    "edges":     ["Minor whitening on bottom edge (back)"],
    "surface":   ["Faint holo scratches visible at angle (front)"],
    "other":     []
  },
  "grading_decision": {
    "gradable_candidate": "maybe",
    "reason": "Card shows light wear consistent with PSA 7-8. Holo scratches are the primary concern.",
    "caveats": []
  }
}

Example when only the front is present (front_only):
{
  "analysis_mode": "front_only",
  "card_identity": { "name": null, "set": null, "year": null, "number": null, "confidence": "low" },
  "image_quality": {
    "status": "partial", "warnings": [], "front_present": true, "back_present": false,
    "centering_visible": true, "corners_visible": true, "edges_visible": true, "surface_visible": false
  },
  "front_analysis": {
    "assessable": true,
    "centering": "55/45 L/R, 53/47 T/B",
    "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] }
  },
  "back_analysis": {
    "assessable": false,
    "centering": null,
    "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] }
  },
  "grade_estimate": {
    "grade_range": "PSA 7-9", "confidence": "low", "limiting_factor": "front_only",
    "distribution": {
      "1": 0.00, "2": 0.00, "3": 0.01, "4": 0.02, "5": 0.05,
      "6": 0.10, "7": 0.22, "8": 0.30, "9": 0.22, "10": 0.08
    }
  },
  "issues": { "centering": [], "corners": [], "edges": [], "surface": [], "other": [] },
  "grading_decision": {
    "gradable_candidate": "maybe",
    "reason": "Front appears reasonably clean but back is not available for assessment.",
    "caveats": ["Back image not provided — rear corner whitening, edge wear, and back centering cannot be assessed. Grade confidence is limited to low."]
  }
}

Rules:
- Output exactly one JSON object. Use exactly the top-level keys shown above.
- front_analysis and back_analysis each have their own "issues" object for per-side defects.
- "issues" at the top level is the COMBINED worst-case from both sides. Tag each item with "(front)" or "(back)".
- Do not guess hidden defects. Only report what is actually visible.
- If the back is missing: set analysis_mode "front_only", back_analysis.assessable false, grade confidence "low", limiting_factor "front_only", and add the caveat string exactly as shown.
- If glare/blur/angle limits visibility, mention in image_quality.warnings and the relevant side analysis.
- Use a wider grade range and lower confidence when evidence is incomplete.
- Be conservative about PSA 9 and PSA 10 if surface or back visibility is limited.
- Distribution values must sum approximately to 1.00.
- confidence "high": top-2 PSA grades >60% of mass. "medium": 40–60%. "low": image too limited."""


print('✅ Prompts loaded')
print(f'  System prompt: {len(SYSTEM_PROMPT):,} chars')
print(f'  User prompt:   {len(USER_PROMPT):,} chars')

## Main Grading Function

Exact port of `gradeWithClaude()` from `claudeVision.ts`.

In [ ]:
def _array_to_base64_jpeg(img_rgb: np.ndarray, max_size: int = 1024,
                           quality: int = 85) -> dict | None:
    """Resize (longest edge ≤ max_size) and encode as JPEG base64."""
    try:
        pil = Image.fromarray(img_rgb)
        w, h = pil.size
        if max(w, h) > max_size:
            scale = max_size / max(w, h)
            pil = pil.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        buf = io.BytesIO()
        pil.save(buf, format='JPEG', quality=quality)
        return {'base64': base64.b64encode(buf.getvalue()).decode(),
                'media_type': 'image/jpeg', 'size': pil.size}
    except Exception as exc:
        print(f'  resize failed: {exc}')
        return None


def grade_with_claude(
    image_urls:      list[str] | None = None,
    local_paths:     list[str] | None = None,
    title:           str  = '',
    api_key:         str  = API_KEY,
    run_centering:   bool = True,   # perspective warp + border scan (front image only)
    run_corners:     bool = True,   # per-corner whitening
    run_scratches:   bool = True,   # border-band Sobel roughness
    show_prompt:     bool = False,
    show_raw:        bool = False,
) -> dict:
    """
    Full grading pipeline — exact port of claudeVision.ts plus Python-only
    centering measurement (perspective warp).

    Pipeline:
      1. Load all images as numpy arrays (shared across all CV steps)
      2. Classify each as front / back / unknown
      3. Sort: fronts first, backs second
      4. Phase 1 CV (quality) on first front image
      5. Phase 2 CV (structure) on first front image:
           a. Corner whitening
           b. Surface roughness (border-band Sobel)
           c. Centering via perspective warp  [Python-only, not in TS backend]
      6. Build image blocks + inject all CV sections into prompt
      7. Call Claude Haiku (max_tokens=2048)
      8. Sanity guard
    """
    client  = anthropic.Anthropic(api_key=api_key)
    sources = image_urls or local_paths or []

    # ── Step 1: Load all images ────────────────────────────────────────────────
    print('  [1/6] Loading images...')
    arrays: list[np.ndarray | None] = [
        (_fetch_buffer(s) if image_urls else _load_buffer(s))
        for s in sources[:6]
    ]

    # ── Step 2: Classify ──────────────────────────────────────────────────────
    print('  [2/6] Classifying front/back...')
    sides = [classify_card_side(a) if a is not None else 'unknown' for a in arrays]
    for src, side in zip(sources[:6], sides):
        print(f'         {Path(src).name if local_paths else src[:50]} → {side}')

    # ── Step 3: Sort ──────────────────────────────────────────────────────────
    rank  = {'front': 0, 'back': 1, 'unknown': 2}
    order = sorted(range(len(arrays)), key=lambda i: rank[sides[i]])
    sorted_arrays = [arrays[i]  for i in order]
    sorted_sides  = [sides[i]   for i in order]
    sorted_srcs   = [sources[i] for i in order]

    # ── Step 4: Phase 1 — image quality ───────────────────────────────────────
    print('  [3/6] Running Phase 1 CV (quality)...')
    front_idx = next((i for i, s in enumerate(sorted_sides) if s == 'front'), 0)
    front_arr = sorted_arrays[front_idx]
    cv_quality = _analyse_image_array(front_arr) if front_arr is not None else None

    # ── Step 5: Phase 2 — structural analysis on front image ──────────────────
    print('  [4/6] Running Phase 2 CV (structure)...')
    corners   = analyse_corners(front_arr)           if (run_corners   and front_arr is not None) else None
    scratch   = analyse_scratch_indicator(front_arr) if (run_scratches and front_arr is not None) else None
    centering = measure_centering(front_arr)         if (run_centering and front_arr is not None) else None

    if centering:
        warp_status = 'warp ✓' if centering['warp_applied'] else 'warp ✗ (fallback)'
        print(f'         centering {centering["lr_ratio"]} L/R, {centering["tb_ratio"]} T/B  [{warp_status}]')
    if corners and corners['any_whitening']:
        print(f'         ⚠ whitening at: {corners["whitening_corners"]}')
    if scratch:
        print(f'         surface roughness: {scratch["severity"]} ({scratch["edge_pixel_count"]} edge px)')

    # ── Step 6: Build image content blocks ────────────────────────────────────
    print('  [5/6] Building image blocks + prompt...')
    image_blocks = []

    if sorted_arrays[0] is not None:
        enc = _array_to_base64_jpeg(sorted_arrays[0])
        if enc:
            image_blocks.append({'type': 'image', 'source': {
                'type': 'base64', 'media_type': enc['media_type'], 'data': enc['base64']}})

    for i, src in enumerate(sorted_srcs[1:], start=1):
        arr = sorted_arrays[i]
        if arr is not None:
            enc = _array_to_base64_jpeg(arr)
            if enc:
                image_blocks.append({'type': 'image', 'source': {
                    'type': 'base64', 'media_type': enc['media_type'], 'data': enc['base64']}})
        elif image_urls:
            image_blocks.append({'type': 'image', 'source': {'type': 'url', 'url': src}})

    # Build prompt with all CV sections
    full_text = (
        format_cv_section(cv_quality)
        + format_corner_section(corners)
        + format_scratch_section(scratch)
        + format_centering_section(centering)
        + format_side_labels(sorted_sides)
        + (f'Listing title (use as identity hint, but trust the image over the title): "{title}"\n\n' if title else '')
        + USER_PROMPT
    )

    if show_prompt:
        print('\n' + '='*70 + '\nSYSTEM PROMPT:\n' + '='*70)
        print(SYSTEM_PROMPT)
        print('\n' + '='*70 + '\nUSER TEXT BLOCK:\n' + '='*70)
        print(full_text)
        print('='*70 + '\n')

    # ── Step 7: Call Claude Haiku ─────────────────────────────────────────────
    print(f'  [6/6] Calling Claude Haiku ({len(image_blocks)} image block(s))...')
    response = client.messages.create(
        model='claude-haiku-4-5', max_tokens=2048, system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': [*image_blocks, {'type': 'text', 'text': full_text}]}]
    )

    raw = ''.join(b.text for b in response.content if b.type == 'text').strip()

    if show_raw:
        print('\n--- RAW RESPONSE ---\n' + raw + '\n--- END RAW ---\n')

    json_str = re.sub(r'^```(?:json)?\s*\n?', '', raw, flags=re.IGNORECASE)
    json_str = re.sub(r'\n?```\s*$', '', json_str).strip()

    try:
        result = json.loads(json_str)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f'Claude returned non-JSON (output_tokens={response.usage.output_tokens}):\n{raw[:600]}'
        ) from exc

    # Normalise distribution
    dist  = result.get('grade_estimate', {}).get('distribution', {})
    total_prob = sum(dist.values())
    if total_prob > 0 and abs(total_prob - 1) > 0.02:
        for k in dist:
            dist[k] = round(dist[k] / total_prob, 4)

    # Sanity guard
    front_ok = result.get('front_analysis', {}).get('assessable', True)
    back_ok  = result.get('back_analysis',  {}).get('assessable', True)
    if not front_ok and not back_ok:
        result['grade_estimate']['confidence']           = 'low'
        result['grade_estimate']['limiting_factor']      = 'image_quality'
        result['grading_decision']['gradable_candidate'] = 'no'
        result.setdefault('grading_decision', {}).setdefault('caveats', []).append(
            'Neither front nor back image was assessable — unable to grade reliably.')
        print('  ⚠ Sanity guard: both sides non-assessable → confidence forced to low')

    # Attach metadata
    result['_cv']        = cv_quality
    result['_corners']   = corners
    result['_scratch']   = scratch
    result['_centering'] = centering
    result['_sides']     = sorted_sides
    result['_usage']     = {'input_tokens':  response.usage.input_tokens,
                             'output_tokens': response.usage.output_tokens}
    return result


print('✅ grade_with_claude() ready (full Phase 1 + 2 CV pipeline)')

## Display Helpers

In [ ]:
GRADE_COLORS = {
    10: '#10b981', 9: '#34d399', 8: '#6366f1',
    7:  '#818cf8', 6: '#f97316', 5: '#fb923c',
    4:  '#ef4444', 3: '#dc2626', 2: '#b91c1c', 1: '#7f1d1d',
}

ISSUE_CATEGORY_LABELS = {
    'centering': 'Centering',
    'corners':   'Corners',
    'edges':     'Edges',
    'surface':   'Surface',
    'other':     'Other',
}


def _print_side_issues(issues: dict) -> bool:
    """Print per-side issues dict; returns True if any issues found."""
    has_any = False
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            has_any = True
            print(f'      {cat.upper()}')
            for item in items:
                print(f'        • {item}')
    return has_any


def print_result(result: dict, label: str = '') -> None:
    """Pretty-print a single grading result to stdout."""
    card   = result.get('card_identity', {})
    iq     = result.get('image_quality', {})
    grade  = result.get('grade_estimate', {})
    issues = result.get('issues', {})
    gd     = result.get('grading_decision', {})
    front  = result.get('front_analysis', {})
    back   = result.get('back_analysis', {})
    cv     = result.get('_cv')
    usage  = result.get('_usage', {})
    mode   = result.get('analysis_mode', 'front_only')

    sep = '=' * 65
    print(f'\n{sep}')
    if label:
        print(f'  {label}')
    print(sep)

    # Card identity
    name    = card.get('name') or '(unknown)'
    cset    = card.get('set') or '?'
    year    = card.get('year') or '?'
    num     = card.get('number') or '?'
    id_conf = card.get('confidence', '?')
    print(f'\n🃏  {name}  [{cset} {year} #{num}]')
    print(f'    ID confidence: {id_conf}   Mode: {mode}')

    # Image quality
    status = iq.get('status', '?').upper()
    print(f'\n📷  Image Quality: {status}')
    chip_defs = [
        ('front_present', 'Front'), ('back_present', 'Back'),
        ('centering_visible', 'Centering'), ('corners_visible', 'Corners'),
        ('edges_visible', 'Edges'),         ('surface_visible', 'Surface'),
    ]
    chips = '  '.join(f"{'✓' if iq.get(k) else '✗'} {lbl}" for k, lbl in chip_defs)
    print(f'    {chips}')
    for w in iq.get('warnings', []):
        print(f'    ⚠ {w}')

    # CV measurements
    if cv:
        bf = '🔴' if cv['is_blurry'] else '🟢'
        gf = '🔴' if cv['has_glare'] else '🟢'
        print(f'\n🔬  CV: blur {cv["blur_score"]:.1f}{bf}  '
              f'glare {cv["glare_fraction"]*100:.1f}%{gf}  '
              f'brightness {cv["brightness_mean"]:.0f}')

    # ── Front Analysis ────────────────────────────────────────────
    print(f'\n── Front Analysis ──────────────────────────────────────────')
    if front.get('assessable') is False:
        print('    ✗ Not assessable — no front image provided')
    else:
        centering = front.get('centering')
        if centering:
            print(f'    Centering: {centering}')
        front_issues = front.get('issues', {})
        if not _print_side_issues({k: v for k, v in front_issues.items() if v}):
            print('    ✓ No issues detected')

    # ── Back Analysis ─────────────────────────────────────────────
    print(f'\n── Back Analysis ───────────────────────────────────────────')
    if back.get('assessable') is False:
        print('    ✗ Not assessable — no back image provided')
    else:
        centering = back.get('centering')
        if centering:
            print(f'    Centering: {centering}')
        back_issues = back.get('issues', {})
        if not _print_side_issues({k: v for k, v in back_issues.items() if v}):
            print('    ✓ No issues detected')

    # ── Grade estimate ────────────────────────────────────────────
    gr   = grade.get('grade_range', '?')
    conf = grade.get('confidence', '?')
    lf   = grade.get('limiting_factor')
    lf_str = f'  [limiting: {lf}]' if lf else ''
    print(f'\n📊  Grade Estimate: {gr}  ({conf} confidence){lf_str}')
    dist = grade.get('distribution', {})
    for g in range(10, 0, -1):
        prob = dist.get(str(g), 0)
        bar  = '█' * int(prob * 32)
        print(f'    PSA {g:2d} │ {bar:<32} {prob*100:5.1f}%')

    # ── Grading decision ──────────────────────────────────────────
    gdc   = gd.get('gradable_candidate', '?')
    emoji = {'yes': '✅', 'maybe': '🤔', 'no': '❌'}.get(gdc, '❓')
    print(f'\n{emoji}  Grading Candidate: {gdc.upper()}')
    print(f'    {gd.get("reason", "")}')

    caveats = gd.get('caveats', [])
    if caveats:
        print(f'\n⛔  Caveats:')
        for c in caveats:
            print(f'    • {c}')

    # ── Combined issues ───────────────────────────────────────────
    print('\n⚠️  Combined Issues (worst-case):')
    has_any = False
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            has_any = True
            print(f'    {cat.upper()}')
            for item in items:
                print(f'      • {item}')
    if not has_any:
        print('    ✓ No significant issues detected')

    # Token usage
    if usage:
        print(f'\n💰  Tokens: {usage.get("input_tokens",0):,} in '
              f'/ {usage.get("output_tokens",0):,} out')
    print(f'\n{sep}\n')


def plot_result(result: dict, label: str = '') -> None:
    """Matplotlib visualisation of a single grading result."""
    grade  = result.get('grade_estimate', {})
    dist   = grade.get('distribution', {})
    issues = result.get('issues', {})
    gd     = result.get('grading_decision', {})
    card   = result.get('card_identity', {})
    iq     = result.get('image_quality', {})
    front  = result.get('front_analysis', {})
    back   = result.get('back_analysis', {})
    mode   = result.get('analysis_mode', '?')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: probability bars ────────────────────────────────────
    ax1 = axes[0]
    grades = list(range(1, 11))
    probs  = [dist.get(str(g), 0) * 100 for g in grades]
    colors = [GRADE_COLORS[g] for g in grades]
    bars   = ax1.bar(grades, probs, color=colors, width=0.7, edgecolor='#30363d')
    ax1.set_xlabel('PSA Grade')
    ax1.set_ylabel('Probability (%)')
    ax1.set_title(
        f'{grade.get("grade_range","?")}  ({grade.get("confidence","?")} confidence)',
        color='white')
    ax1.set_xticks(grades)
    ax1.set_ylim(0, max(probs) * 1.15 + 1)
    ax1.spines['bottom'].set_color('#30363d')
    ax1.spines['left'].set_color('#30363d')
    for bar, prob in zip(bars, probs):
        if prob > 2:
            ax1.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.4,
                     f'{prob:.0f}%', ha='center', va='bottom',
                     color='white', fontsize=8)

    # ── Right: summary text ───────────────────────────────────────
    ax2 = axes[1]
    ax2.axis('off')
    gdc   = gd.get('gradable_candidate', '?')
    emoji = {'yes': '✅', 'maybe': '🤔', 'no': '❌'}.get(gdc, '?')

    # Build front/back summary strings
    def side_summary(side: dict, name: str) -> list[str]:
        lines = [f'{name}:']
        if not side.get('assessable', True):
            lines.append('  ✗ not assessable')
            return lines
        c = side.get('centering')
        if c:
            lines.append(f'  centering {c}')
        side_issues = side.get('issues', {})
        count = sum(len(v) for v in side_issues.values() if isinstance(v, list))
        lines.append(f'  {count} issue(s) detected')
        return lines

    lines = [
        f"Card: {card.get('name') or 'Unknown'}",
        f"Set:  {card.get('set') or '?'}  {card.get('year') or ''}",
        f"ID confidence: {card.get('confidence', '?')}",
        f"Mode: {mode}   IQ: {iq.get('status','?')}",
        "",
    ]
    lines += side_summary(front, 'Front')
    lines += side_summary(back, 'Back')
    lines += [
        "",
        f"{emoji} Grading: {gdc.upper()}",
        textwrap.fill(gd.get('reason', ''), 42),
    ]
    caveats = gd.get('caveats', [])
    if caveats:
        lines += ["", "⛔ Caveats:"]
        for c in caveats:
            lines.append(f"  • {textwrap.shorten(c, 40)}")
    lines += ["", "Combined Issues:"]
    for cat in ['centering', 'corners', 'edges', 'surface', 'other']:
        items = issues.get(cat, [])
        if items:
            lines.append(f"  {cat.upper()} ({len(items)})")
            for it in items:
                lines.append(f"    • {textwrap.shorten(it, 36)}")
    if not any(issues.get(c) for c in ['centering','corners','edges','surface','other']):
        lines.append('  ✓ No significant issues')

    ax2.text(0.04, 0.97, '\n'.join(lines),
             transform=ax2.transAxes, color='white',
             fontsize=9.5, va='top', family='monospace', linespacing=1.55)
    ax2.set_title(label or 'Assessment', color='white')

    plt.suptitle(label, color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


print('✅ Display helpers ready')

---
## Single Card Test

Paste eBay image URLs (s-l1600) **or** supply local file paths.  
Set `show_prompt=True` to print the exact prompt sent to Claude — useful when debugging prompt changes.

In [ ]:
# ── Configure your test card here ────────────────────────────────────────────

# Option A: eBay image URLs  (replace with real URLs)
TEST_IMAGE_URLS = [
    'https://i.ebayimg.com/images/g/REPLACE_ME/s-l1600.jpg',
    # add more URLs for multi-image tests
]

# Option B: Local files  (comment out TEST_IMAGE_URLS above and use this)
# TEST_LOCAL_PATHS = ['image0_front.jpeg', 'image0_back.jpeg']
TEST_LOCAL_PATHS = None

TEST_TITLE = 'Charizard ex 199/165 Scarlet & Violet 151 — PSA?'
TEST_PRICE = 480.0

SHOW_PROMPT = False   # True → print the full prompt sent to Claude
SHOW_RAW    = False   # True → print Claude's raw string before JSON parse

In [ ]:
# Show thumbnails first
if TEST_LOCAL_PATHS:
    show_images(local_paths=TEST_LOCAL_PATHS)
elif 'REPLACE_ME' not in TEST_IMAGE_URLS[0]:
    show_images(image_urls=TEST_IMAGE_URLS)

# Run grading
print('\nRunning grading pipeline...')
result = grade_with_claude(
    image_urls=TEST_IMAGE_URLS  if not TEST_LOCAL_PATHS else None,
    local_paths=TEST_LOCAL_PATHS if TEST_LOCAL_PATHS     else None,
    title=TEST_TITLE,
    show_prompt=SHOW_PROMPT,
    show_raw=SHOW_RAW,
)

print_result(result, label=TEST_TITLE)

In [ ]:
plot_result(result, label=TEST_TITLE)

In [ ]:
# Inspect the raw parsed JSON (excluding base64 image data)
display_json = {k: v for k, v in result.items() if not k.startswith('_')}
print(json.dumps(display_json, indent=2, ensure_ascii=False))

---
## Consistency Test — Same Card, N Runs

Run the same card multiple times to measure how consistent Claude's output is across calls.
Useful for understanding grade range variance before committing to a prompt.

In [ ]:
N_RUNS = 3   # number of times to repeat the same card

repeat_results = []
for i in tqdm(range(N_RUNS), desc='Consistency runs'):
    r = grade_with_claude(
        image_urls=TEST_IMAGE_URLS  if not TEST_LOCAL_PATHS else None,
        local_paths=TEST_LOCAL_PATHS if TEST_LOCAL_PATHS     else None,
        title=TEST_TITLE,
    )
    r['_run'] = i + 1
    repeat_results.append(r)

# Summary table
rows = []
for r in repeat_results:
    grade = r.get('grade_estimate', {})
    dist  = grade.get('distribution', {})
    gd    = r.get('grading_decision', {})
    rows.append({
        'Run':        r['_run'],
        'Grade Range': grade.get('grade_range', '?'),
        'Confidence':  grade.get('confidence', '?'),
        'Gradable':    gd.get('gradable_candidate', '?'),
        'P(PSA10)':   f"{dist.get('10',0)*100:.1f}%",
        'P(PSA9)':    f"{dist.get('9',0)*100:.1f}%",
        'P(PSA8)':    f"{dist.get('8',0)*100:.1f}%",
        'P(PSA7)':    f"{dist.get('7',0)*100:.1f}%",
        'In$':        r.get('_usage', {}).get('input_tokens', 0),
        'Out$':       r.get('_usage', {}).get('output_tokens', 0),
    })

df_repeat = pd.DataFrame(rows)
print(df_repeat.to_string(index=False))

In [ ]:
# Overlay distributions from all runs
fig, ax = plt.subplots(figsize=(12, 5))

grades = list(range(1, 11))
x = np.arange(len(grades))
width = 0.8 / N_RUNS
cmap = plt.cm.get_cmap('tab10', N_RUNS)

for idx, r in enumerate(repeat_results):
    dist  = r.get('grade_estimate', {}).get('distribution', {})
    probs = [dist.get(str(g), 0) * 100 for g in grades]
    offset = (idx - N_RUNS / 2 + 0.5) * width
    ax.bar(x + offset, probs, width=width * 0.9,
           label=f'Run {idx+1}', color=cmap(idx), alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'PSA {g}' for g in grades])
ax.set_ylabel('Probability (%)')
ax.set_title(f'Grade Distribution Consistency — {N_RUNS} runs', color='white')
ax.legend()
ax.spines['bottom'].set_color('#30363d')
ax.spines['left'].set_color('#30363d')
plt.tight_layout()
plt.show()

---
## Batch Test — Multiple Cards

Define a list of test cases below (name, URLs or local paths, title, known grade if available) and run them all in one shot.

In [ ]:
# ── Define test cases ─────────────────────────────────────────────────────────
# Each dict keys:
#   name         : short label for display
#   image_urls   : list of eBay image URLs  (or use local_paths)
#   local_paths  : list of local image paths (comment out image_urls key)
#   title        : listing title (used as identity hint only)
#   price        : listing price in USD
#   known_grade  : actual PSA grade if you have it — used for accuracy calc (optional)

BATCH_CASES = [
    {
        'name':       'Card A',
        'local_paths': ['image0_front.jpeg', 'image0_back.jpeg'],
        'title':       'Card A — test',
        'price':       50.0,
        'known_grade': None,
    },
    {
        'name':       'Card B',
        'local_paths': ['image1_front.jpeg', 'image1_back.jpeg'],
        'title':       'Card B — test',
        'price':       30.0,
        'known_grade': None,
    },
    # Add more cases — paste eBay URLs or local paths
]

In [ ]:
batch_results = []

for i, tc in enumerate(BATCH_CASES):
    print(f"\n[{i+1}/{len(BATCH_CASES)}] {tc['name']}...")
    try:
        r = grade_with_claude(
            image_urls=tc.get('image_urls'),
            local_paths=tc.get('local_paths'),
            title=tc.get('title', ''),
        )
        r['_test_name']   = tc['name']
        r['_price']       = tc.get('price', 0)
        r['_known_grade'] = tc.get('known_grade')
        batch_results.append(r)
        print_result(r, label=tc['name'])
    except Exception as exc:
        print(f'  ❌ Error: {exc}')
        batch_results.append({
            '_test_name': tc['name'],
            '_error': str(exc)
        })

print(f'\n✅ Batch complete: {sum(1 for r in batch_results if "_error" not in r)}/{len(BATCH_CASES)} succeeded')

In [ ]:
# Comparison table
rows = []
for r in batch_results:
    if '_error' in r:
        rows.append({'Name': r['_test_name'], 'Error': r['_error']})
        continue
    grade  = r.get('grade_estimate', {})
    dist   = grade.get('distribution', {})
    card   = r.get('card_identity', {})
    gd     = r.get('grading_decision', {})
    iq     = r.get('image_quality', {})
    cv     = r.get('_cv') or {}
    front  = r.get('front_analysis', {})
    back   = r.get('back_analysis', {})
    known  = r.get('_known_grade')

    # Compute expected PSA from distribution
    ev = sum(int(g) * p for g, p in dist.items())

    rows.append({
        'Name':         r.get('_test_name', '?'),
        'Card':         card.get('name') or '?',
        'Mode':         r.get('analysis_mode', '?'),
        'Range':        grade.get('grade_range', '?'),
        'Conf':         grade.get('confidence', '?'),
        'Limit':        grade.get('limiting_factor') or '—',
        'E[PSA]':       f'{ev:.1f}',
        'Known':        known or '—',
        'Gradable':     gd.get('gradable_candidate', '?'),
        'Caveats':      len(gd.get('caveats', [])),
        'F.assess':     '✓' if front.get('assessable') else '✗',
        'B.assess':     '✓' if back.get('assessable') else '✗',
        'IQ':           iq.get('status', '?'),
        'Blur':         f"{cv.get('blur_score', '?')}",
        'Glare%':       f"{cv.get('glare_fraction', 0)*100:.1f}",
        'Issues':       sum(len(r.get('issues', {}).get(c, [])) for c in ['centering','corners','edges','surface','other']),
        'In tok':       r.get('_usage', {}).get('input_tokens', 0),
        'Out tok':      r.get('_usage', {}).get('output_tokens', 0),
    })

df_batch = pd.DataFrame(rows)
print(df_batch.to_string(index=False))

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results to plot')
else:
    n = len(valid)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, valid):
        dist   = r.get('grade_estimate', {}).get('distribution', {})
        probs  = [dist.get(str(g), 0) * 100 for g in range(1, 11)]
        colors = [GRADE_COLORS[g] for g in range(1, 11)]
        bars   = ax.bar(range(1, 11), probs, color=colors, width=0.7, edgecolor='#30363d')
        gr     = r.get('grade_estimate', {}).get('grade_range', '?')
        ax.set_title(f"{r.get('_test_name','?')}\n{gr}", color='white', fontsize=10)
        ax.set_xticks(range(1, 11))
        ax.set_xlabel('PSA Grade')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')
        for bar, prob in zip(bars, probs):
            if prob > 5:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.5,
                        f'{prob:.0f}%', ha='center', va='bottom',
                        color='white', fontsize=8)

    axes[0].set_ylabel('Probability (%)')
    plt.suptitle('Batch Grade Distributions', color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Accuracy Analysis

If you supplied `known_grade` for any test cases, this section measures how well Claude's expected-value estimate matches the actual PSA grade.

In [ ]:
labelled = [
    r for r in batch_results
    if '_error' not in r and r.get('_known_grade') is not None
]

if not labelled:
    print('No labelled test cases (set known_grade in BATCH_CASES to enable accuracy analysis)')
else:
    errors, abs_errors = [], []
    for r in labelled:
        dist  = r.get('grade_estimate', {}).get('distribution', {})
        ev    = sum(int(g) * p for g, p in dist.items())
        known = float(r['_known_grade'])
        err   = ev - known
        errors.append(err)
        abs_errors.append(abs(err))
        print(f"  {r.get('_test_name','?'):20s}  E[PSA]={ev:.1f}  Actual={known:.0f}  Error={err:+.1f}")

    print(f'\n  MAE:  {np.mean(abs_errors):.2f} PSA grades')
    print(f'  Bias: {np.mean(errors):+.2f} (positive = overestimates grade)')
    print(f'  Within ±1 grade: {sum(1 for e in abs_errors if e <= 1)}/{len(abs_errors)}')
    print(f'  Within ±2 grade: {sum(1 for e in abs_errors if e <= 2)}/{len(abs_errors)}')

---
## Issues Frequency Analysis

Aggregates which issue categories appear most often across your batch — useful for spotting systematic bias in the prompt.

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results')
else:
    cats = ['centering', 'corners', 'edges', 'surface', 'other']
    counts = {c: 0 for c in cats}
    total_issues = 0
    for r in valid:
        for cat in cats:
            n = len(r.get('issues', {}).get(cat, []))
            counts[cat] += n
            total_issues += n

    fig, ax = plt.subplots(figsize=(9, 4))
    cat_labels = [c.capitalize() for c in cats]
    bar_colors = ['#6366f1', '#f97316', '#10b981', '#ef4444', '#9ca3af']
    bars = ax.bar(cat_labels, [counts[c] for c in cats],
                  color=bar_colors, edgecolor='#30363d')
    for bar, cat in zip(bars, cats):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                str(counts[cat]),
                ha='center', va='bottom', color='white', fontsize=11)
    ax.set_ylabel('Total issues reported')
    ax.set_title(f'Issue category frequency — {len(valid)} card(s), {total_issues} total issues',
                 color='white')
    ax.spines['bottom'].set_color('#30363d')
    ax.spines['left'].set_color('#30363d')
    plt.tight_layout()
    plt.show()

    print('\nAll reported issues:')
    for cat in cats:
        all_items = []
        for r in valid:
            all_items.extend(r.get('issues', {}).get(cat, []))
        if all_items:
            print(f'\n  {cat.upper()}')
            for item in all_items:
                print(f'    \u2022 {item}')

---
## Side Analysis Summary

Per-side (front vs back) issue frequency breakdown and assessability overview across the batch.

In [ ]:
valid = [r for r in batch_results if '_error' not in r]
if not valid:
    print('No valid results')
else:
    cats = ['centering', 'corners', 'edges', 'surface', 'other']

    # ── Assessability summary ─────────────────────────────────────
    front_assess = sum(1 for r in valid if r.get('front_analysis', {}).get('assessable', True))
    back_assess  = sum(1 for r in valid if r.get('back_analysis',  {}).get('assessable', True))
    front_only   = sum(1 for r in valid if r.get('analysis_mode') == 'front_only')
    front_back   = sum(1 for r in valid if r.get('analysis_mode') == 'front_back')
    print(f'Cards analysed: {len(valid)}')
    print(f'  front_back mode : {front_back}')
    print(f'  front_only mode : {front_only}')
    print(f'  Front assessable: {front_assess}/{len(valid)}')
    print(f'  Back  assessable: {back_assess}/{len(valid)}')

    # ── Per-side issue counts ─────────────────────────────────────
    front_counts = {c: 0 for c in cats}
    back_counts  = {c: 0 for c in cats}
    for r in valid:
        for cat in cats:
            front_counts[cat] += len(r.get('front_analysis', {}).get('issues', {}).get(cat, []))
            back_counts[cat]  += len(r.get('back_analysis',  {}).get('issues', {}).get(cat, []))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    cat_labels = [c.capitalize() for c in cats]
    bar_colors = ['#6366f1', '#f97316', '#10b981', '#ef4444', '#9ca3af']

    for ax, counts, title in [
        (axes[0], front_counts, 'Front Issues'),
        (axes[1], back_counts,  'Back Issues'),
    ]:
        bars = ax.bar(cat_labels, [counts[c] for c in cats],
                      color=bar_colors, edgecolor='#30363d')
        for bar, cat in zip(bars, cats):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.05,
                    str(counts[cat]),
                    ha='center', va='bottom', color='white', fontsize=11)
        ax.set_title(title, color='white')
        ax.set_ylabel('Issue count')
        ax.spines['bottom'].set_color('#30363d')
        ax.spines['left'].set_color('#30363d')

    plt.suptitle('Front vs Back Issue Breakdown', color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Caveats summary ───────────────────────────────────────────
    caveats_total = sum(len(r.get('grading_decision', {}).get('caveats', [])) for r in valid)
    if caveats_total:
        print(f'\n⛔  {caveats_total} caveat(s) across {len(valid)} card(s):')
        for r in valid:
            for c in r.get('grading_decision', {}).get('caveats', []):
                print(f'  [{r.get("_test_name","?")}] {c}')
    else:
        print('\n✓  No caveats — all cards had back images available')